In [8]:
#importlibraries

#libraries to extract frame
import os
import pandas as pd
import cv2
import numpy as np
import shutil
import pickle as pk
import random

#libraries to undergo differencing
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from torchvision.utils import save_image
from sklearn.decomposition import PCA

#create a function that transforms images to tensors
convert_tensor = transforms.ToTensor()
convert_image = transforms.ToPILImage()

#set directories
#Imgae_to_video_locator_csv
#image_to_vid_file = '../../data/Deepfish/location_data/location_dat.csv'
#image_to_vid_file = '../../data/Kakadu_fish/location_data/location_dat.csv'
image_to_vid_file = '../../data/Tassy_BRUV/location_data/location_dat.csv'
#training data location
#folder with images, labels and yaml file are assumed to be within training data folder
#training_dat_loc = '../../data/Deepfish/training_data/'
#training_dat_loc = '../../data/Kakadu_fish/training_data/'
training_dat_loc = '../../data/Tassy_BRUV/training_data/'

#whether to read in center frame from surrounding video/images
center_image_surround = True

#directory where differenced images will be saved
#diff_folder = '../../data/Deepfish/augmented_images/'
diff_folder = '../../data/Tassy_BRUV/augmented_images/'

In [9]:
#images
image_loc = training_dat_loc + 'images/'

#import image location data
val_frame_dat = pd.read_csv(image_to_vid_file)

In [10]:
im_num_list_full = list(range(0,val_frame_dat.shape[0]))

In [11]:
im_num_list_train = []

for im_num in im_num_list_full:
    if(val_frame_dat['split'][im_num]=="train"):
        im_num_list_train.append(im_num)

In [15]:
len(im_num_list_train)

1340

In [14]:
im_num_list_train = [im_num for im_num in im_num_list_full if val_frame_dat['split'][im_num] == "train"]

In [16]:
im_num_list = random.sample(im_num_list_train, 10)

In [17]:
im_num_list

[255, 262, 453, 1369, 854, 1548, 792, 1411, 1234, 496]

In [7]:
im_num_list = list(range(0,val_frame_dat.shape[0]))
#im_num_list = list(range(0,5))

for im_num in im_num_list:
    
    if(val_frame_dat['split'][im_num]=="train"):
    
        #read in center image
        if(center_image_surround==True):
                #read in video
            video = val_frame_dat['vid_location'][im_num]
            vidcap = cv2.VideoCapture(video)
            frame3 = val_frame_dat['frame'][im_num]
            vidcap.set(1, frame3)
            success,img_center = vidcap.read()
        else:
            img_center = Image.open(image_loc + val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])

        #convert images to tensors
        img_center_tens = convert_tensor(img_center)

        #fix video image before differencing
        if(center_image_surround==True):
            img_center_tens = torch.stack((img_center_tens[2,:,:],img_center_tens[1,:,:],img_center_tens[0,:,:]),dim=0)
        
        
        im_depth, im_height, im_width,  = img_center_tens.shape    
        # Reshape the image tensor into a 2D matrix
        im_matrix = np.reshape(convert_image(img_center_tens), (im_height * im_width, im_depth))
        
        if(im_num==0):
            #start off with the first image matrix
            im_matrix_all = im_matrix
        else:
            im_matrix_all = np.concatenate((im_matrix_all,im_matrix))

pca = PCA(n_components=2,whiten=True)
pca.fit(im_matrix_all)
            
pk.dump(pca, open(diff_folder + "pca.pkl","wb"))
pd.DataFrame(pca.explained_variance_ratio_).to_csv(diff_folder+"explained_var.csv")
# later reload the pickle file
#pca_reload = pk.load(open("pca.pkl",'rb'))
#result_new = pca_reload .transform(X)

KeyboardInterrupt: 